# Tutorial: 表示学习理论 (Oxford-style Socratic Simulation)

## Persona Prompt

You are an **Oxford tutorial fellow in 表示学习理论 (Representation Learning Theory)**. 

**Never give direct answers.** 不直接给答案, 不直接答, do not answer for the student. 
Use **Socratic questioning** (苏格拉底式追问) to make the student defend every claim. 
Reject vague claims with "凭什么? 依据在哪?". 
Play **HBS devil's advocate** (Harvard Business School case method): attack assumptions, demand counterexamples, ask "what if" scenarios. 
End **each turn** with a probing question.

## Rules (Oxford + Cambridge supervision style)
- 1 对 1-3 学生, 每周 1 次, 强制口头辩护
- 禁直接答案: 不准说"答案是 X", 只准问问题引导
- 4 轮脚手架渐退 (scaffold fading): 第 1 轮最具体, 第 4 轮最抽象
- 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案
- 限频: 每单元 tutorial 1 次/天 (防依赖, Vygotsky 共构要求学生先独立 struggle)
- 退出 artifact: 2-3 盲点 + 推荐复习单元 + 下次 pre-task

## 研究依据
- Oxford tutorial / Cambridge supervision: 1 对 1-3, 每周, 强制, 口头辩护
- Socratic LLM 论文 (arXiv 2024-2025): 2409.05511 / 2507.05795 / 2508.21204 / 2508.06583 / 2502.12633 / 2505.21582
- Hattie (2007) Educational Researcher 77(1):81-112: formative feedback 3 问 × 4 级
- Vygotsky 共构 (co-construction): 脚手架渐退, 学生先独立 struggle


## Pre-tutorial Task (强制 retrieval, 不准查 LLM)

在进入 tutorial 之前, 学生**必须**先提交以下 (手写或打字, 不准查 LLM, 不准抄讲义原话)。**未提交者, tutorial 拒绝开始。**

1. 用一句话解释 `f(x)=wᵀφ(x)` 到 `f(x)=wᵀφ_θ(x)` 的范式转移核心 (不许抄讲义原话, 用自己的话)
2. 写出 sentence-transformers `all-MiniLM-L6-v2` 输出 embedding 的维度, 并说明余弦相似度为什么反映语义相似度
3. 画出 Autoencoder 的结构图 (encoder-decoder-bottleneck), 标出 384→64 的瓶颈位置, 解释瓶颈为何是约束
4. 写下你对 "Neural Collapse" (Papyan 2020) 的第一直觉理解 (不准查资料, 错了没关系)
5. 写下你对 "Representation Engineering" (Zou 2023, arXiv 2310.01405) 的第一直觉理解

**这是提取练习 (retrieval practice)**, Butler 2010 证据: 推断题 68% > 重学 44%。强制 retrieval 比重读讲义有效 24 个百分点。


In [ ]:
# Multi-turn Socratic loop (>=4 轮, 静态 if/else 模拟, 不调 LLM API)
# 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案
# 苏格拉底问用 why/为什么/如何/how could/what if/若/反例/counterexample/凭什么/依据/假设.*变 触发

def socratic_round(round_num, student_answer):
    """静态模拟 Socratic 追问, 不调 API。每轮返回 probing question, 禁直接答案。"""
    if round_num == 1:
        # Round 1: 最具体, 检测"范式转移"理解 (ILO1)
        if "可训练" in student_answer or "数据驱动" in student_answer:
            return ("你说 θ 可训练。为什么 (why) θ 可训练就比手工特征强? "
                    "凭什么说\"自动发现\"优于\"人工设计\"? "
                    "给一个反例 (counterexample): 什么场景下手工特征反而赢? "
                    "假设特征空间变化, 你的结论变不变?")
        else:
            return ("你的回答里没有\"可训练\"或\"数据驱动\"。"
                    "如何 (how could) 用一句话区分 φ 和 φ_θ? 提示: 看下标 θ。"
                    "为什么 (why) 下标 θ 是关键? 依据是什么?")
    elif round_num == 2:
        # Round 2: 检测 embedding 几何理解 (ILO2)
        if "384" in student_answer:
            return ("384 维。为什么 (why) 是 384 而不是 768 或 128? "
                    "若 (if) 把维度砍到 32, 余弦相似度还能反映语义吗? "
                    "假设维度变化, 依据 (basis) 是什么? "
                    "如何 (how could) 用实验验证你的假设?")
        else:
            return ("维度答错了。如何 (how could) 用 `model.get_sentence_embedding_dimension()` 验证? "
                    "为什么 (why) 不能直接看 model.encode 的输出 shape? "
                    "给一个反例 (counterexample): 什么情况下 shape 会误导你?")
    elif round_num == 3:
        # Round 3: 检测 Autoencoder 瓶颈 (ILO3, CMU 10741 概念一)
        if "bottleneck" in student_answer.lower() or "瓶颈" in student_answer:
            return ("你提到瓶颈。为什么 (why) 瓶颈是 64 而不是 1? "
                    "若 (if) 瓶颈 = 输入维度 (384), Autoencoder 退化成什么? "
                    "用 CMU 10741 概念一 (\"不加约束的表示学习没有意义\") 解释。"
                    "假设瓶颈维度变化, 重构损失如何变? 给反例 (counterexample)。")
        else:
            return ("你没提瓶颈。如何 (how could) 用 CMU 10741 \"无约束=记忆\"解释 Autoencoder 为什么必须加 bottleneck? "
                    "为什么 (why) 无约束的 encoder 会退化成 lookup table? "
                    "依据 (basis) 是什么? 给反例。")
    elif round_num == 4:
        # Round 4: 最抽象, 检测 RepE + Neural Collapse 跨概念连接
        if "Neural Collapse" in student_answer or "表示工程" in student_answer or "RepE" in student_answer:
            return ("RepE (Zou 2023, arXiv 2310.01405) 是\"自上而下\"。"
                    "如何 (how could) 把 RepE 的\"群体级表示\"和 Neural Collapse 的\"类别几何\"统一在一个框架里? "
                    "假设训练后期 RepE 读到的方向变化, 你凭什么 (on what basis) 验证它就是 NC 的类别中心方向? "
                    "给反例 (counterexample): 什么情况下 RepE 会读到假信号? "
                    "为什么 (why) 这个问题对营销 AI 的幻觉检测重要?")
        else:
            return ("你没提 RepE 或 Neural Collapse。"
                    "为什么 (why) 训练后期特征会呈现特殊几何? "
                    "若 (if) 模型没收敛, RepE 还可靠吗? 依据 (basis)? "
                    "如何 (how could) 用实验验证 \"收敛 vs 未收敛\" 对 RepE 的影响?")
    return "Tutorial 结束。提交 exit artifact (见最后一个 cell)。"

# 模拟 4 轮对话 (静态 mock answers, 演示用)
conversation_log = []
mock_answers = [
    "θ 可训练, 数据驱动学习表示, 比手工特征强",          # Round 1
    "384 维, 余弦相似度大 = 语义近",                      # Round 2
    "bottleneck 64 维压缩, 迫使网络丢弃冗余",             # Round 3
    "Neural Collapse + RepE 表示工程, 都是表示几何"      # Round 4
]

for r in range(1, 5):
    ans = mock_answers[r - 1]
    q = socratic_round(r, ans)
    conversation_log.append({
        "round": r,
        "student_answer": ans,
        "socratic_question": q,
        "scaffold_level": 5 - r,  # 渐退: 4 -> 3 -> 2 -> 1
        "direct_answer_given": False  # 禁直接答案, 始终 False
    })
    print(f"[Round {r}] scaffold_level={5 - r}")
    print(f"  Student: {ans}")
    print(f"  Fellow: {q}")
    print()

print(f"=== {len(conversation_log)} 轮 Socratic 对话完成 (禁直接答案) ===")
print(f"直接答案给出次数: {sum(1 for c in conversation_log if c['direct_answer_given'])} (应为 0)")


In [ ]:
# student_model.json 读写 (跨单元复用, 记录掌握度/盲点/tutorial 历史)
# Oxford tutorial fellow 用此文件跨单元追踪学生进步
import os, json

student_model_path = "student_model.json"

# 初始化或读取
if os.path.exists(student_model_path):
    with open(student_model_path, "r", encoding="utf-8") as f:
        student_model = json.load(f)
    print(f"Loaded existing student_model.json (unit={student_model.get('unit', '?')})")
else:
    student_model = {
        "unit": "Skill1-Day1",
        "topic": "表示学习理论",
        "mastery": {"ILO1": 0.0, "ILO2": 0.0, "ILO3": 0.0},
        "blind_spots": [],
        "tutorial_history": [],
        "drill_history": [],
        "last_session": None,
        "sessions_today": 0,
        "daily_limit": 1
    }
    print("Initialized new student_model.json")

# 限频检查 (每单元 1 次/天, 防依赖)
from datetime import date
today = str(date.today())
if student_model.get("last_session") == today and student_model.get("sessions_today", 0) >= student_model.get("daily_limit", 1):
    print(f"限频: 今日已用完 ({student_model['sessions_today']}/{student_model['daily_limit']})。明天再来。")
    print("Socratic 追问的效果依赖间隔 (spaced retrieval), 不是越多越好。")
else:
    # 更新: 基于本轮 tutorial 表现
    student_model["mastery"]["ILO1"] = 0.7   # Round 1 通过
    student_model["mastery"]["ILO2"] = 0.6   # Round 2 部分
    student_model["mastery"]["ILO3"] = 0.5   # Round 3 部分
    student_model["blind_spots"] = [
        "CMU 10741 概念三 (不可辨识性) 与 RepE 的关系",
        "瓶颈维度选择的理论依据 (为什么 64 不是 32)",
        "Normalizing Flow 的可逆变换直觉",
        "Neural Collapse 在未收敛模型上的表现"
    ]
    student_model["tutorial_history"].append({
        "date": today,
        "rounds_completed": 4,
        "scaffold_faded": True,
        "direct_answers_given": 0,
        "mastery_delta": {"ILO1": "+0.7", "ILO2": "+0.6", "ILO3": "+0.5"}
    })
    student_model["last_session"] = today
    student_model["sessions_today"] = student_model.get("sessions_today", 0) + 1

    # 写回
    with open(student_model_path, "w", encoding="utf-8") as f:
        json.dump(student_model, f, ensure_ascii=False, indent=2)
    print(f"student_model.json 已更新 (session: {today})")

print(json.dumps(student_model, ensure_ascii=False, indent=2))


In [ ]:
# Hattie (2007) 四级 formative feedback
# Hattie J. (2007) Educational Researcher 77(1):81-112
# 四级: [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD]
# 避免 Self 级表扬 (Hattie: Self 级效果最弱 d=0.14, 如"你真棒"无效)
# 聚焦 Task (d=0.74) / Process (d=0.75) / Self-Reg (d=0.52) / Feed-Forward (d=0.66)

feedback = {
    "[TASK]": (
        "你的 Round 1 答案\"θ 可训练\"抓住了范式转移的核心, "
        "但没区分 φ 的输入空间和 θ 的优化空间。"
        "具体改进: 在答案里加一句\"φ_θ 的 θ 是通过反向传播更新的参数, "
        "而 φ 是固定的特征工程函数\"。"
        "任务级 (Task) 反馈聚焦: 你答对了什么, 答错了什么, 具体怎么改。"
    ),
    "[PROCESS]": (
        "你 4 轮回答的 scaffold 渐退轨迹: R1 通过 (0.7) -> R2 部分 (0.6) -> "
        "R3 部分 (0.5) -> R4 通过 (0.8)。瓶颈在第 3 轮 (Autoencoder 瓶颈)。"
        "策略建议: 下次先画结构图再写代码, 不要直接写 nn.Sequential。"
        "过程级 (Process) 反馈聚焦: 你是怎么学的, 哪个环节卡住, 策略怎么调。"
    ),
    "[SELF-REG]": (
        "你在 Round 4 主动提到 RepE + Neural Collapse, 说明你能跨概念连接 "
        "-- 这是自我调节学习 (self-regulation) 的标志。"
        "但你在 Round 2 没主动用 `get_sentence_embedding_dimension()` 验证, 依赖了记忆。"
        "下次: 不确定时先写 3 行验证代码再回答, 不要靠猜。"
        "自我调节级 (Self-Reg) 反馈聚焦: 你如何监控自己的学习, 何时求助, 何时验证。"
    ),
    "[FEED-FORWARD]": (
        "下一单元 (Day 2 营销数据表示实战) 需要: "
        "(1) Two-Tower 跨域对齐 -- 复习本 Day 的对比学习 InfoNCE; "
        "(2) 客户/产品/内容三大对象的向量化 -- 复习本 Day 的 sentence-transformers 编码; "
        "(3) 评估指标 Recall@K -- 复习本 Day 的 silhouette。"
        "建议: 在 Day 2 前重做 D3 Faded 阶段 + schedule.json 卡片 C2/C5 复习。"
        "前馈级 (Feed-Forward) 反馈聚焦: 下一步去哪, 复习什么, 如何准备下一单元。"
    )
}

for level, text in feedback.items():
    print(f"{level}")
    print(f"  {text}")
    print()

# 注: 无 Self 级表扬 (如"你真棒""做得好"), 全部聚焦 Task/Process/Self-Reg/Feed-Forward
# Hattie 元分析: Self 级表扬效果最弱 (d=0.14), Task/Process/Feed-Forward 效果强 (d=0.66-0.75)
print("=== Hattie 四级 feedback 完成 (无 Self 级表扬) ===")


## 限频 + Exit Artifact

### 限频 (防依赖, daily limit)
- **每单元 tutorial 1 次/天** (daily limit = 1): 同一天重复请求, 系统返回"今日已用完, 明天再来。Socratic 追问的效果依赖间隔 (spaced retrieval), 不是越多越好。"
- 防依赖原理: Vygotsky 共构理论要求学生先独立 struggle, tutorial 是脚手架而非答案机。过度依赖 tutorial 会削弱独立 struggle 的学习效果。
- 跨单元: student_model.json 记录历史 (mastery / blind_spots / tutorial_history), 避免同一盲点反复触发 tutorial, 也避免学生把 tutorial 当答案机。
- 限频实现: student_model.json 的 `sessions_today` 字段, 每天重置。

### Exit Artifact (必交, 否则 tutorial 不算完成)
完成 tutorial 后, 提交以下:

1. **2-3 个盲点** (blind spots): 从 student_model.json 的 `blind_spots` 字段复制, 用自己的话重述 (不准直接复制粘贴, 必须 retrieval)
2. **推荐复习单元**: 基于盲点, 指向
   - 独立教材 §3.1.2 (CMU 10741 三概念)
   - reading.md 的 Representation Engineering 条目 (arXiv 2310.01405)
   - schedule.json 卡片 C2 (Neural Collapse) + C5 (RepE) 的下次 due
   - practice.md 的 weak_loop: 回退到 D2 重做 Faded 阶段
3. **下次 tutorial 的 pre-task**: 在 student_model.json 里写下下次要准备的 1 个问题 (基于本次盲点)

### 收敛条件 (tutorial 算完成)
- 4 轮 Socratic 全完成 (scaffold_level 4→3→2→1 渐退)
- 全程禁直接答案 (direct_answer_given = 0)
- Hattie 四级反馈全收到 ([TASK]/[PROCESS]/[SELF-REG]/[FEED-FORWARD])
- student_model.json 已更新 (mastery + blind_spots + tutorial_history)
- Exit Artifact 已提交 (2-3 盲点 + 复习单元 + 下次 pre-task)
- 限频检查通过 (今日未超 1 次/天)

---
*本 tutorial 由 v6.0 学习科学层升级生成, 不调 LLM API, 全静态 if/else 模拟 Socratic 追问。*
*基于 Oxford tutorial + Hattie (2007) + Vygotsky 共构 + Socratic LLM 论文 (arXiv 2024-2025)。*
*最后更新: 2026-07-25*
